In [144]:
# ============================================================
# MuM 重建效果可视化 — 模仿 test.py 使用 mum 包
# ============================================================

import os, sys, torch, glob
import numpy as np
from matplotlib import pyplot as plt

# 添加 MuM 包路径（模仿 test.py）
sys.path.insert(0, '/data/data_taohy/modelReShow/MuM')

from mum.model import vit_base
from mum.model import vit_small
from mum.utils import transform_image
from mum.utils.viz import qualitative_evaluation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"设备: {device}")

设备: cuda


In [145]:
# ============================================================
# 设置 checkpoint 路径（在此修改！）
# ============================================================

# 可填文件名（自动拼接 checkpoint_dir）或完整路径
CKPT_PATH = "mum_0730-133650_final_step15000.pth"

checkpoint_dir = "/data/data_taohy/modelReShow/mum/checkpoints"

# 列出可用 checkpoint 供参考
checkpoints = sorted(glob.glob(os.path.join(checkpoint_dir, "*.pth")))
print(f"可用 checkpoint ({len(checkpoints)} 个):")
for ckpt in checkpoints:
    size_mb = os.path.getsize(ckpt) / (1024 * 1024)
    tag = "  ← final" if "final" in os.path.basename(ckpt) else ""
    model_type = "vit_small" if os.path.getsize(ckpt) < 500 * 1024 * 1024 else "vit_base"
    print(f"  {os.path.basename(ckpt)}  ({size_mb:.0f} MB)  {model_type}{tag}")

# 解析路径
if os.path.isabs(CKPT_PATH):
    ckpt_path = CKPT_PATH
else:
    ckpt_path = os.path.join(checkpoint_dir, CKPT_PATH)

assert os.path.exists(ckpt_path), f"checkpoint 不存在: {ckpt_path}"
print(f"\n加载: {os.path.basename(ckpt_path)}")

可用 checkpoint (11 个):
  mum_0730-020235_step1800.pth  (1278 MB)  vit_base
  mum_0730-020235_step25200.pth  (1278 MB)  vit_base
  mum_0730-020235_step9000.pth  (1278 MB)  vit_base
  mum_0730-104027_step34200.pth  (1278 MB)  vit_base
  mum_0730-133650_final_step15000.pth  (244 MB)  vit_small  ← final
  mum_0730-133650_step14400.pth  (244 MB)  vit_small
  mum_0730-133650_step1800.pth  (244 MB)  vit_small
  mum_0730-154103_final_step15000.pth  (244 MB)  vit_small  ← final
  mum_0730-154103_step10800.pth  (244 MB)  vit_small
  mum_0730-154103_step14400.pth  (244 MB)  vit_small
  mum_0730-154103_step1800.pth  (244 MB)  vit_small

加载: mum_0730-133650_final_step15000.pth


In [146]:
# ============================================================
# 构建模型并加载自训练权重（自动检测 vit_small / vit_base）
# ============================================================

# 根据 checkpoint 大小自动判断模型类型
ckpt_size = os.path.getsize(ckpt_path)
if ckpt_size < 500 * 1024 * 1024:  # < 500MB → vit_small
    model = vit_small(patch_size=16, img_size=224, norm_pix_loss=True)
    model_type = "vit_small"
else:
    model = vit_base(patch_size=16, img_size=224, norm_pix_loss=True)
    model_type = "vit_base"

print(f"模型类型: {model_type}  (ckpt={ckpt_size/(1024**2):.0f} MB)")

state_dict = torch.load(ckpt_path, map_location=device, weights_only=True)
if "model_state_dict" in state_dict:
    state_dict = state_dict["model_state_dict"]
elif "model" in state_dict:
    state_dict = state_dict["model"]

# 映射 checkpoint key → 模型 key（ckpt 用 encoder/decoder 包装了子模块）
def map_key(k: str) -> str:
    k = k.replace("module.", "")
    k = k.replace("encoder.", "")
    # decoder.rope_embed → rope_embed_decoder（必须在 decoder. 剥离之前）
    if 'decoder.rope_embed' in k:
        k = k.replace('decoder.rope_embed', 'rope_embed_decoder')
    else:
        k = k.replace("decoder.", "")
    return k

new_sd = {map_key(k): v for k, v in state_dict.items()}

missing, unexpected = model.load_state_dict(new_sd, strict=False)
if missing:
    print(f"⚠ 缺失 key ({len(missing)}个): {missing[:3]}...")
if unexpected:
    print(f"⚠ 多余 key ({len(unexpected)}个): {unexpected[:3]}...")
if not missing and not unexpected:
    print("✅ 权重加载成功！")  

模型类型: vit_small  (ckpt=244 MB)
✅ 权重加载成功！


In [149]:
# ============================================================
# 加载图片（模仿 test.py 使用 transform_image）
# ============================================================

img_paths = sorted(glob.glob("/data/data_taohy/datasets/BlendedMVS/*/blended_images/*.jpg"))
img_paths = [p for p in img_paths if "_masked" not in p]
sample_paths = img_paths[:3]  # 取 3 张

print("图片路径:")
for p in sample_paths:
    print(f"  {p}")

# transform_image 自动 Resize + Normalize（与 test.py 一致）
imgs = torch.stack(
    [transform_image(p, size=(224, 224)) for p in sample_paths]
).unsqueeze(0)  # [1, S, 3, 224, 224]

print(f"\n输入形状: {imgs.shape} (B={imgs.shape[0]}, S={imgs.shape[1]})")

图片路径:
  /data/data_taohy/datasets/BlendedMVS/57f8d9bbe73f6760f10e916a/blended_images/00000000.jpg
  /data/data_taohy/datasets/BlendedMVS/57f8d9bbe73f6760f10e916a/blended_images/00000001.jpg
  /data/data_taohy/datasets/BlendedMVS/57f8d9bbe73f6760f10e916a/blended_images/00000002.jpg

输入形状: torch.Size([1, 3, 3, 224, 224]) (B=1, S=3)


In [148]:
# ============================================================
# 定性评估 — 使用 mum 包自带的 qualitative_evaluation
# 自动显示：原图 | Masked | 重建 三栏对比
# ============================================================
import functools

MASK_RATIO = 0.7  # 可调：0.25 / 0.5 / 0.75 / 0.9

# 确保 imgs 在相同设备上
imgs = imgs.to(device)

# 临时覆盖 forward 的默认 mask_ratio，传给 qualitative_evaluation
original_forward = model.forward
model.forward = functools.partial(original_forward, mask_ratio=MASK_RATIO)

qualitative_evaluation(
    model,
    imgs,
    path="/data/data_taohy/modelReShow/mum/reconstruction.png",
    visible=True,  # True: 可见 patch 保留原图, 只重建被遮住的 patch
)

model.forward = original_forward  # 恢复
print("重建可视化完成！图片已保存到 reconstruction.png")

with torch.inference_mode():
    loss, pred, mask = model(imgs, mask_ratio=MASK_RATIO)
    print(f"重建 Loss (MSE, norm_pix): {loss.item():.6f}")
    print(f"实际 mask 比例: {mask.float().mean().item():.3f}")

RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same